# 🛡️ Risk Report – Quant Analyst
**Workflow**: Multi-Asset Returns → Correlation Heatmap → VaR & CVaR → Tail Risk Distribution

---

In [5]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

session = AnalystSession(role=RoleContext.ANALYST)

SYMBOLS = ['BTC-USD', 'ETH-USD']
DAYS = 365
CONFIDENCE = 0.95

PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. Portfolio Data Prep
Aggregating returns for a diversified crypto portfolio.

In [6]:
import polars as pl

def _load_returns(session, sym: str, days: int) -> pl.DataFrame:
    """Load returns for a symbol, normalizing timestamp to UTC-naive to allow joins."""
    try:
        df_asset = session.load_ohlcv(sym, '1d')
        if len(df_asset) < 10:
            raise ValueError(f"Only {len(df_asset)} rows")
    except Exception as e:
        print(f"  [{sym}] Fallback to synthetic: {e}")
        df_asset = session.sample_ohlcv(symbol=sym, days=days)
    df_asset = session.make_returns(df_asset)
    # Normalize timestamp: strip timezone so join always succeeds
    ts = df_asset["timestamp"]
    if ts.dtype != pl.Datetime("us"):
        # Cast to UTC-naive by removing timezone info
        df_asset = df_asset.with_columns(
            pl.col("timestamp").dt.convert_time_zone("UTC").dt.replace_time_zone(None)
        )
    return df_asset.select(["timestamp", "returns"]).rename({"returns": sym})

returns_data = []
for sym in SYMBOLS:
    print(f"Loading {sym}...")
    returns_data.append(_load_returns(session, sym, DAYS))

# Join and convert to numpy
merged = returns_data[0]
for other in returns_data[1:]:
    merged = merged.join(other, on='timestamp', how='inner')

merged = merged.drop_nulls()
R = merged.select(SYMBOLS).to_numpy()
print(f"Joined Returns shape: {R.shape}")

Loading BTC-USD...
Loading ETH-USD...
Joined Returns shape: (729, 2)


## 2. Dynamic Correlation Heatmap
Visualizing inter-asset dependencies via Plotly.

In [7]:
corr = np.corrcoef(R.T)
fig_corr = px.imshow(
    corr, x=SYMBOLS, y=SYMBOLS, 
    color_continuous_scale='RdBu_r', 
    zmin=-1, zmax=1,
    title="Portfolio Asset Correlation"
)
fig_corr.update_layout(**PLOTLY_DARK)
fig_corr.show()

## 3. VaR & CVaR Analysis
Quantifying the expected loss in the worst-case scenario.

In [8]:
w = np.ones(len(SYMBOLS)) / len(SYMBOLS)
port_returns = R @ w

var_val = np.percentile(port_returns, (1 - CONFIDENCE) * 100)
cvar_val = port_returns[port_returns <= var_val].mean()

fig_dist = px.histogram(port_returns, nbins=100, title="Portfolio Return Distribution & Tail Risk")
fig_dist.add_vline(x=var_val, line_dash="dash", line_color="orange", annotation_text=f"VaR {CONFIDENCE*100:.0f}%")
fig_dist.add_vline(x=cvar_val, line_dash="dot", line_color="red", annotation_text=f"CVaR {CONFIDENCE*100:.0f}%")
fig_dist.update_layout(**PLOTLY_DARK)
fig_dist.show()

print(f"95% VaR: {var_val:.4f} | 95% CVaR: {cvar_val:.4f}")

/var/folders/h9/_37f97bx28q57hg0p4dlwxcc0000gn/T/ipykernel_42372/518657705.py:2: RuntimeWarning: divide by zero encountered in matmul
  port_returns = R @ w
/var/folders/h9/_37f97bx28q57hg0p4dlwxcc0000gn/T/ipykernel_42372/518657705.py:2: RuntimeWarning: overflow encountered in matmul
  port_returns = R @ w
/var/folders/h9/_37f97bx28q57hg0p4dlwxcc0000gn/T/ipykernel_42372/518657705.py:2: RuntimeWarning: invalid value encountered in matmul
  port_returns = R @ w


95% VaR: -0.0471 | 95% CVaR: -0.0669
